In [ ]:
# ============================================================
# Part 1 - Imports
# ============================================================

import os
import time
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.distributions import Normal

from models.model import Vision2Drive


# ======
from metadrive import MetaDriveEnv

warnings.filterwarnings("ignore")

plt.style.use("ggplot")

In [ ]:
# ============================================================
# Part 2 - Reinforcement Learning Configuration
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    "bc_checkpoint": "checkpoints/best_model.pth",
    "save_dir": "rl_checkpoints",

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    "episodes": 1000,
    "max_steps": 1000,

    "learning_rate": 3e-4,
    "weight_decay": 1e-5,

    # --------------------------------------------------------
    # PPO Hyperparameters
    # --------------------------------------------------------

    "gamma": 0.99,
    "gae_lambda": 0.95,

    "clip_epsilon": 0.2,

    "ppo_epochs": 10,
    "mini_batch_size": 64,

    "value_loss_coef": 0.5,
    "entropy_coef": 0.01,

    "max_grad_norm": 0.5,

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    "eval_interval": 20,
    "save_interval": 50,

    # --------------------------------------------------------
    # Environment
    # --------------------------------------------------------

    "use_render": False,
    "num_scenarios": 1000,

    "traffic_density": 0.1,

    "vehicle_config": {
        "lidar": {
            "num_lasers": 240,
            "distance": 50
        }
    }

}

In [ ]:
# ============================================================
# Part 3 - Device & Reproducibility
# ============================================================

SEED = 42


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.manual_seed(seed)


set_seed(SEED)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")

elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")

else:
    DEVICE = torch.device("cpu")

print(f"Using Device : {DEVICE}")
print(f"Random Seed  : {SEED}")

<!-- Option A (Recommended):

Vision2Drive Backbone
        │
        ├── PPO Policy Head
        └── PPO Value Head

This reuses your entire pretrained multimodal backbone and only replaces the final heads.

Option B:

Keep the original action head exactly as it is and add only a value head. -->

In [ ]:
# ============================================================
# Part 4 - Build PPO Vision2Drive
# ============================================================

# PPO Actor Head
# ============================================================

class PPOActor(nn.Module):
    """
    PPO Policy Network.

    Predicts the mean and standard deviation of the
    continuous driving actions.

    Actions:
        - Steering
        - Throttle
        - Brake
    """

    def __init__(self, feature_dim=512, action_dim=3):
        super().__init__()

        self.policy = nn.Sequential(

            nn.Linear(feature_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, action_dim)

        )

        # Learnable log standard deviation
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, features):

        mean = self.policy(features)

        std = torch.exp(self.log_std)

        return mean, std


# ============================================================
# PPO Critic Head
# ============================================================

class PPOCritic(nn.Module):
    """
    State Value Network.
    """

    def __init__(self, feature_dim=512):
        super().__init__()

        self.value = nn.Sequential(

            nn.Linear(feature_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 1)

        )

    def forward(self, features):

        return self.value(features)


# ============================================================
# PPO Vision2Drive
# ============================================================

class PPOVision2Drive(nn.Module):
    """
    PPO version of Vision2Drive.

    Uses the pretrained Vision2Drive backbone and
    adds PPO Actor and Critic heads.
    """

    def __init__(self):

        super().__init__()

        # --------------------------------------------------
        # Behavior Cloning Backbone
        # --------------------------------------------------

        self.backbone = Vision2Drive()

        # Remove BC Driving Head
        self.backbone.head = nn.Identity()

        # --------------------------------------------------
        # PPO Heads
        # --------------------------------------------------

        self.actor = PPOActor()

        self.critic = PPOCritic()

    def forward(
        self,
        rgb,
        lidar_bev,
        vehicle_state,
        navigation,
    ):

        # --------------------------------------------------
        # Backbone Features
        # --------------------------------------------------

        features = self.backbone.extract_features(
            rgb,
            lidar_bev,
            vehicle_state,
            navigation,
        )

        # --------------------------------------------------
        # Actor
        # --------------------------------------------------

        mean, std = self.actor(features)

        # --------------------------------------------------
        # Critic
        # --------------------------------------------------

        value = self.critic(features)

        return {

            "mean": mean,
            "std": std,
            "value": value

        }

    @torch.no_grad()
    def act(
        self,
        rgb,
        lidar_bev,
        vehicle_state,
        navigation,
    ):

        outputs = self.forward(
            rgb,
            lidar_bev,
            vehicle_state,
            navigation,
        )

        dist = Normal(
            outputs["mean"],
            outputs["std"],
        )

        action = dist.sample()

        log_prob = dist.log_prob(action).sum(dim=-1)

        return action, log_prob, outputs["value"]

In [ ]:
# ============================================================
# Part 5 - Load Behavior Cloning Weights
# ============================================================

print("=" * 60)
print("Initializing PPO Vision2Drive")
print("=" * 60)

# ------------------------------------------------------------
# Create PPO Model
# ------------------------------------------------------------

model = PPOVision2Drive().to(DEVICE)

# ------------------------------------------------------------
# Load Behavior Cloning Checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(

    CONFIG["bc_checkpoint"],

    map_location=DEVICE,

)

# ------------------------------------------------------------
# Load Pretrained Backbone
# ------------------------------------------------------------

missing_keys, unexpected_keys = model.backbone.load_state_dict(

    checkpoint["model_state_dict"],

    strict=False,

)

print("\nBehavior Cloning weights loaded successfully.")

print(f"Missing Keys    : {len(missing_keys)}")
print(f"Unexpected Keys : {len(unexpected_keys)}")

print("\nActor Head    : Randomly Initialized")
print("Critic Head   : Randomly Initialized")

print("\nBackbone      : Loaded from BC Checkpoint")

print("=" * 60)

In [ ]:
# ============================================================
# Part 6 - MetaDrive Environment
# ============================================================

from metadrive import MetaDriveEnv


# ------------------------------------------------------------
# MetaDrive Configuration
# ------------------------------------------------------------

ENV_CONFIG = {

    # Number of parallel maps
    "num_scenarios": 100,

    # Random traffic
    "traffic_density": 0.10,

    # Enable navigation
    "use_render": False,

    # RGB Camera
    "image_observation": True,

    "image_source": "rgb_camera",

    "camera_height": 224,

    "camera_width": 224,

    # LiDAR
    "vehicle_config": {

        "lidar": {

            "num_lasers": 240,

            "distance": 50,

        }

    },

    # Episode Length
    "horizon": CONFIG["max_steps"],

    # Random Seed
    "start_seed": CONFIG["seed"],

}


# ------------------------------------------------------------
# Create Environment
# ------------------------------------------------------------

env = MetaDriveEnv(ENV_CONFIG)

print("=" * 60)
print("MetaDrive Environment Initialized")
print("=" * 60)

print(f"Scenarios        : {ENV_CONFIG['num_scenarios']}")
print(f"Traffic Density  : {ENV_CONFIG['traffic_density']}")
print(f"Episode Length   : {ENV_CONFIG['horizon']}")
print(f"Camera Size      : 224 x 224")
print(f"LiDAR Distance   : 50 meters")
print("=" * 60)

In [ ]:
# ============================================================
# Part 7 - Observation Processing
# ============================================================

import numpy as np
import torch
import torchvision.transforms as transforms


# ------------------------------------------------------------
# RGB Transform
# ------------------------------------------------------------

rgb_transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485, 0.456, 0.406],

        std=[0.229, 0.224, 0.225],

    )

])


# ------------------------------------------------------------
# Observation Processor
# ------------------------------------------------------------

def process_observation(observation):
    """
    Convert MetaDrive observations into Vision2Drive inputs.

    Returns
    -------
    rgb
    lidar_bev
    vehicle_state
    navigation
    """

    # --------------------------------------------------
    # RGB Camera
    # --------------------------------------------------

    rgb = observation["image"]

    rgb = rgb_transform(rgb)

    rgb = rgb.unsqueeze(0).to(DEVICE)

    # --------------------------------------------------
    # LiDAR BEV
    # --------------------------------------------------

    lidar = observation["lidar"]

    lidar = torch.tensor(

        lidar,

        dtype=torch.float32,

        device=DEVICE,

    )

    lidar = lidar.unsqueeze(0)

    # --------------------------------------------------
    # Vehicle State
    # --------------------------------------------------

    speed = observation["vehicle_state"]["speed"]

    steering = observation["vehicle_state"]["steering"]

    vehicle_state = torch.tensor(

        [[speed, steering]],

        dtype=torch.float32,

        device=DEVICE,

    )

    # --------------------------------------------------
    # Navigation Command
    # --------------------------------------------------

    navigation = observation["navigation"]

    navigation = torch.tensor(

        navigation,

        dtype=torch.float32,

        device=DEVICE,

    ).unsqueeze(0)

    return {

        "rgb": rgb,

        "lidar": lidar,

        "vehicle_state": vehicle_state,

        "navigation": navigation,

    }

In [ ]:
# ============================================================
# Part 8 - Reward Function
# ============================================================

def compute_reward(info):
    """
    Compute the PPO reward from MetaDrive information.

    Parameters
    ----------
    info : dict
        Information returned by MetaDrive after each step.

    Returns
    -------
    float
        Total reward.
    """

    reward = 0.0

    # --------------------------------------------------------
    # Forward Progress
    # --------------------------------------------------------

    reward += info.get("progress", 0.0)

    # --------------------------------------------------------
    # Lane Keeping
    # --------------------------------------------------------

    reward += 0.5 * info.get("on_lane_reward", 0.0)

    # --------------------------------------------------------
    # Speed Reward
    # --------------------------------------------------------

    reward += 0.1 * info.get("speed_reward", 0.0)

    # --------------------------------------------------------
    # Collision Penalty
    # --------------------------------------------------------

    if info.get("crash_vehicle", False):
        reward -= 20.0

    if info.get("crash_object", False):
        reward -= 10.0

    if info.get("crash_building", False):
        reward -= 20.0

    # --------------------------------------------------------
    # Off Road
    # --------------------------------------------------------

    if info.get("out_of_road", False):
        reward -= 20.0

    # --------------------------------------------------------
    # Wrong Direction
    # --------------------------------------------------------

    if info.get("wrong_way", False):
        reward -= 10.0

    # --------------------------------------------------------
    # Success Bonus
    # --------------------------------------------------------

    if info.get("arrive_dest", False):
        reward += 100.0

    return reward

In [ ]:
# ============================================================
# Part 9 - Episode Initialization
# ============================================================

def initialize_episode(env):
    """
    Reset MetaDrive and prepare the initial observation.

    Parameters
    ----------
    env : MetaDriveEnv

    Returns
    -------
    processed_obs : dict
        Processed observation for Vision2Drive.

    episode_reward : float

    episode_step : int
    """

    observation, info = env.reset()

    processed_obs = process_observation(observation)

    episode_reward = 0.0

    episode_step = 0

    return (
        processed_obs,
        episode_reward,
        episode_step,
        info,
    )

In [ ]:
# ------------------------------------------------------------
# Test Episode Initialization
# ------------------------------------------------------------

obs, episode_reward, episode_step, info = initialize_episode(env)

print("=" * 60)
print("Episode Initialized Successfully")
print("=" * 60)

print("RGB Shape           :", obs["rgb"].shape)
print("LiDAR Shape         :", obs["lidar"].shape)
print("Vehicle State Shape :", obs["vehicle_state"].shape)
print("Navigation Shape    :", obs["navigation"].shape)

print("\nEpisode Reward :", episode_reward)
print("Episode Step   :", episode_step)

In [ ]:
# ============================================================
# Part 10 - Action Selection
# ============================================================

from torch.distributions import Normal


def select_action(
    model,
    observation,
):
    """
    Select an action using the PPO policy.

    Parameters
    ----------
    model : PPOVision2Drive

    observation : dict

    Returns
    -------
    action : np.ndarray

    log_prob : torch.Tensor

    value : torch.Tensor
    """

    model.eval()

    with torch.no_grad():

        outputs = model(

            observation["rgb"],

            observation["lidar"],

            observation["vehicle_state"],

            observation["navigation"],

        )

    # --------------------------------------------------------
    # Create Gaussian Policy
    # --------------------------------------------------------

    distribution = Normal(

        outputs["mean"],

        outputs["std"],

    )

    # --------------------------------------------------------
    # Sample Action
    # --------------------------------------------------------

    action = distribution.sample()

    # --------------------------------------------------------
    # Log Probability
    # --------------------------------------------------------

    log_prob = distribution.log_prob(action)

    log_prob = log_prob.sum(dim=-1)

    # --------------------------------------------------------
    # State Value
    # --------------------------------------------------------

    value = outputs["value"]

    # --------------------------------------------------------
    # Convert to NumPy
    # --------------------------------------------------------

    action = action.squeeze(0)

    action = action.cpu().numpy()

    # Steering

    action[0] = np.clip(

        action[0],

        -1.0,

        1.0,

    )

    # Throttle

    action[1] = np.clip(

        action[1],

        0.0,

        1.0,

    )

    # Brake

    action[2] = np.clip(

        action[2],

        0.0,

        1.0,

    )

    return (

        action,

        log_prob,

        value,

    )

In [ ]:
# ============================================================
# Part 11 - Environment Step
# ============================================================

def environment_step(
    env,
    action,
):
    """
    Execute one interaction step in MetaDrive.

    Parameters
    ----------
    env : MetaDriveEnv

    action : np.ndarray

    Returns
    -------
    processed_observation

    reward

    terminated

    truncated

    info
    """

    # --------------------------------------------------------
    # Execute Action
    # --------------------------------------------------------

    next_observation, _, terminated, truncated, info = env.step(
        action
    )

    # --------------------------------------------------------
    # Compute Custom Reward
    # --------------------------------------------------------

    reward = compute_reward(info)

    # --------------------------------------------------------
    # Process Observation
    # --------------------------------------------------------

    processed_observation = process_observation(
        next_observation
    )

    return (

        processed_observation,

        reward,

        terminated,

        truncated,

        info,

    )

In [ ]:
# ------------------------------------------------------------
# Test Action Selection + Environment Step
# ------------------------------------------------------------

observation, episode_reward, episode_step, info = initialize_episode(env)

action, log_prob, value = select_action(
    model,
    observation,
)

print("=" * 60)
print("Sampled Action")
print("=" * 60)

print(f"Steering : {action[0]:.3f}")
print(f"Throttle : {action[1]:.3f}")
print(f"Brake    : {action[2]:.3f}")

next_observation, reward, terminated, truncated, info = environment_step(
    env,
    action,
)

print("\nReward      :", reward)
print("Terminated :", terminated)
print("Truncated  :", truncated)

In [ ]:
# ============================================================
# Part 12 - Trajectory Buffer
# ============================================================

class TrajectoryBuffer:
    """
    Stores one rollout collected from the environment.
    """

    def __init__(self):
        self.clear()

    def clear(self):
        self.observations = []
        self.actions = []
        self.log_probs = []
        self.values = []
        self.rewards = []
        self.dones = []

    def store(self, observation, action, log_prob, value, reward, done):
        self.observations.append(observation)
        self.actions.append(torch.tensor(action, dtype=torch.float32))
        self.log_probs.append(log_prob.detach())
        self.values.append(value.detach())
        self.rewards.append(reward)
        self.dones.append(done)

    def size(self):
        return len(self.rewards)

In [ ]:
buffer = TrajectoryBuffer()

print("=" * 60)
print("Trajectory Buffer Initialized")
print("=" * 60)
print(f"Current Buffer Size : {buffer.size()}")

In [ ]:
# ============================================================
# Part 13 - Return Calculation
# ============================================================

def compute_returns(rewards, dones, gamma):
    """
    Compute discounted returns for a trajectory.
    """

    returns = []
    discounted_return = 0.0

    for reward, done in zip(reversed(rewards), reversed(dones)):

        if done:
            discounted_return = 0.0

        discounted_return = reward + gamma * discounted_return
        returns.insert(0, discounted_return)

    return torch.tensor(
        returns,
        dtype=torch.float32,
        device=DEVICE,
    )

In [ ]:
sample_rewards = [1, 1, 1, 1]
sample_dones = [False, False, False, True]

returns = compute_returns(
    sample_rewards,
    sample_dones,
    CONFIG["gamma"],
)

print("=" * 60)
print("Discounted Returns")
print("=" * 60)
print(returns)

In [ ]:
# ============================================================
# Part 14 - Generalized Advantage Estimation (GAE)
# ============================================================

def compute_gae(rewards, values, dones, gamma, gae_lambda):
    """
    Compute Generalized Advantage Estimation (GAE).
    """

    advantages = []
    gae = 0.0

    values = values.squeeze(-1).cpu().numpy().tolist()
    values.append(0.0)

    for step in reversed(range(len(rewards))):

        mask = 1.0 - float(dones[step])

        delta = rewards[step] + gamma * values[step + 1] * mask - values[step]

        gae = delta + gamma * gae_lambda * mask * gae

        advantages.insert(0, gae)

    advantages = torch.tensor(
        advantages,
        dtype=torch.float32,
        device=DEVICE,
    )

    advantages = (advantages - advantages.mean()) / (
        advantages.std() + 1e-8
    )

    return advantages

In [ ]:
sample_values = torch.tensor([[0.5], [0.7], [0.6], [0.3]])

advantages = compute_gae(
    sample_rewards,
    sample_values,
    sample_dones,
    CONFIG["gamma"],
    CONFIG["gae_lambda"],
)

print("=" * 60)
print("Generalized Advantage Estimation")
print("=" * 60)
print(advantages)

In [ ]:
# ============================================================
# Part 15 - PPO Loss
# ============================================================

def compute_ppo_loss(
    new_log_probs,
    old_log_probs,
    advantages,
    values,
    returns,
    entropy,
    clip_epsilon,
    value_loss_coef,
    entropy_coef,
):
    """
    Compute PPO loss.
    """

    ratio = torch.exp(new_log_probs - old_log_probs)

    clipped_ratio = torch.clamp(
        ratio,
        1.0 - clip_epsilon,
        1.0 + clip_epsilon,
    )

    actor_loss = -torch.min(
        ratio * advantages,
        clipped_ratio * advantages,
    ).mean()

    critic_loss = (returns - values.squeeze(-1)).pow(2).mean()

    entropy_loss = entropy.mean()

    total_loss = (
        actor_loss
        + value_loss_coef * critic_loss
        - entropy_coef * entropy_loss
    )

    return {
        "total_loss": total_loss,
        "actor_loss": actor_loss,
        "critic_loss": critic_loss,
        "entropy_loss": entropy_loss,
    }

In [ ]:
batch_size = 8

new_log_probs = torch.randn(batch_size)
old_log_probs = torch.randn(batch_size)

advantages = torch.randn(batch_size)
returns = torch.randn(batch_size)

values = torch.randn(batch_size, 1)
entropy = torch.rand(batch_size)

losses = compute_ppo_loss(
    new_log_probs=new_log_probs,
    old_log_probs=old_log_probs,
    advantages=advantages,
    values=values,
    returns=returns,
    entropy=entropy,
    clip_epsilon=CONFIG["clip_epsilon"],
    value_loss_coef=CONFIG["value_loss_coef"],
    entropy_coef=CONFIG["entropy_coef"],
)

print("=" * 60)
print("PPO Loss")
print("=" * 60)

for name, value in losses.items():
    print(f"{name:15s}: {value.item():.4f}")

In [ ]:
# ============================================================
# Part 16 - PPO Optimizer
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

print("=" * 60)
print("Optimizer Initialized")
print("=" * 60)
print(f"Optimizer     : Adam")
print(f"Learning Rate : {CONFIG['learning_rate']}")
print(f"Weight Decay  : {CONFIG['weight_decay']}")

In [ ]:
# ============================================================
# Part 17 - PPO Update Function
# ============================================================

def ppo_update(model, optimizer, buffer):
    """
    Perform one PPO optimization step.
    """

    # --------------------------------------------------------
    # Prepare Training Targets
    # --------------------------------------------------------

    returns = compute_returns(
        buffer.rewards,
        buffer.dones,
        CONFIG["gamma"],
    )

    values = torch.cat(buffer.values)

    advantages = compute_gae(
        buffer.rewards,
        values,
        buffer.dones,
        CONFIG["gamma"],
        CONFIG["gae_lambda"],
    )

    # --------------------------------------------------------
    # Stack Buffer Data
    # --------------------------------------------------------

    rgb = torch.cat([obs["rgb"] for obs in buffer.observations])
    lidar = torch.cat([obs["lidar"] for obs in buffer.observations])
    vehicle = torch.cat([obs["vehicle_state"] for obs in buffer.observations])
    navigation = torch.cat([obs["navigation"] for obs in buffer.observations])

    actions = torch.stack(buffer.actions).to(DEVICE)
    old_log_probs = torch.stack(buffer.log_probs).to(DEVICE)

    # --------------------------------------------------------
    # PPO Optimization
    # --------------------------------------------------------

    model.train()

    for _ in range(CONFIG["ppo_epochs"]):

        outputs = model(
            rgb,
            lidar,
            vehicle,
            navigation,
        )

        distribution = Normal(
            outputs["mean"],
            outputs["std"],
        )

        new_log_probs = distribution.log_prob(actions).sum(dim=-1)
        entropy = distribution.entropy().sum(dim=-1)

        losses = compute_ppo_loss(
            new_log_probs=new_log_probs,
            old_log_probs=old_log_probs,
            advantages=advantages,
            values=outputs["value"],
            returns=returns,
            entropy=entropy,
            clip_epsilon=CONFIG["clip_epsilon"],
            value_loss_coef=CONFIG["value_loss_coef"],
            entropy_coef=CONFIG["entropy_coef"],
        )

        optimizer.zero_grad()

        losses["total_loss"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["max_grad_norm"],
        )

        optimizer.step()

    return losses

In [ ]:
print("=" * 60)
print("PPO Update Function Ready")
print("=" * 60)

print(f"PPO Epochs        : {CONFIG['ppo_epochs']}")
print(f"Clip Epsilon      : {CONFIG['clip_epsilon']}")
print(f"Max Gradient Norm : {CONFIG['max_grad_norm']}")

In [ ]:
# ============================================================
# Part 18 - Evaluation Function
# ============================================================

def evaluate(model, env, episodes=5):
    """
    Evaluate the PPO agent.
    """

    model.eval()

    total_reward = 0.0

    for _ in range(episodes):

        observation, episode_reward, _, _ = initialize_episode(env)

        done = False

        while not done:

            action, _, _ = select_action(model, observation)

            observation, reward, terminated, truncated, _ = environment_step(
                env,
                action,
            )

            episode_reward += reward
            done = terminated or truncated

        total_reward += episode_reward

    average_reward = total_reward / episodes

    return average_reward

In [ ]:
print("=" * 60)
print("Evaluation Function Ready")
print("=" * 60)

avg_reward = evaluate(model, env, episodes=1)

print(f"Average Reward : {avg_reward:.2f}")

In [ ]:
# ============================================================
# Part 19 - Complete RL Training Loop
# ============================================================

print("=" * 60)
print("Starting PPO Training")
print("=" * 60)

training_rewards = []
evaluation_rewards = []

for episode in range(CONFIG["episodes"]):

    # --------------------------------------------------------
    # Initialize Episode
    # --------------------------------------------------------

    observation, episode_reward, _, _ = initialize_episode(env)

    buffer = TrajectoryBuffer()

    done = False

    # --------------------------------------------------------
    # Collect Rollout
    # --------------------------------------------------------

    while not done:

        action, log_prob, value = select_action(
            model,
            observation,
        )

        next_observation, reward, terminated, truncated, _ = environment_step(
            env,
            action,
        )

        done = terminated or truncated

        buffer.store(
            observation,
            action,
            log_prob,
            value,
            reward,
            done,
        )

        observation = next_observation
        episode_reward += reward

    # --------------------------------------------------------
    # PPO Update
    # --------------------------------------------------------

    losses = ppo_update(
        model,
        optimizer,
        buffer,
    )

    training_rewards.append(episode_reward)

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    if (episode + 1) % CONFIG["eval_interval"] == 0:

        avg_reward = evaluate(
            model,
            env,
            episodes=5,
        )

        evaluation_rewards.append(avg_reward)

        print(
            f"[Episode {episode+1:5d}/{CONFIG['episodes']}] "
            f"Reward: {episode_reward:8.2f} | "
            f"Eval: {avg_reward:8.2f} | "
            f"Actor: {losses['actor_loss']:.4f} | "
            f"Critic: {losses['critic_loss']:.4f}"
        )

    # --------------------------------------------------------
    # Save Checkpoint
    # --------------------------------------------------------

    if (episode + 1) % CONFIG["save_interval"] == 0:

        torch.save(

            {
                "episode": episode + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },

            f"{CONFIG['save_dir']}/ppo_episode_{episode+1}.pth",

        )

print("\nTraining Complete.")

In [ ]:
# ============================================================
# Part 20 - Training Visualization
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

plt.plot(training_rewards, label="Training Reward", linewidth=2)

if len(evaluation_rewards) > 0:
    eval_x = range(
        CONFIG["eval_interval"],
        CONFIG["episodes"] + 1,
        CONFIG["eval_interval"],
    )
    plt.plot(eval_x, evaluation_rewards, label="Evaluation Reward", linewidth=2)

plt.title("PPO Training Progress")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(actor_losses, label="Actor Loss")
plt.plot(critic_losses, label="Critic Loss")

plt.title("PPO Loss Curves")
plt.xlabel("Update")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
# ============================================================
# Part 21 - Live MetaDrive Demonstration
# ============================================================

demo_env = MetaDriveEnv({
    **ENV_CONFIG,
    "use_render": True,
})

observation, _, _, _ = initialize_episode(demo_env)

done = False
episode_reward = 0.0

model.eval()

while not done:

    action, _, _ = select_action(model, observation)

    observation, reward, terminated, truncated, _ = environment_step(
        demo_env,
        action,
    )

    episode_reward += reward
    done = terminated or truncated

demo_env.close()

print("=" * 60)
print("Live Demonstration Complete")
print("=" * 60)
print(f"Episode Reward : {episode_reward:.2f}")

In [ ]:
# ============================================================
# Part 22 - Save Final RL Model
# ============================================================

import os

os.makedirs(CONFIG["save_dir"], exist_ok=True)

save_path = os.path.join(
    CONFIG["save_dir"],
    "vision2drive_ppo_final.pth",
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": CONFIG,
        "training_rewards": training_rewards,
        "evaluation_rewards": evaluation_rewards,
    },
    save_path,
)

print("=" * 60)
print("PPO Training Completed Successfully")
print("=" * 60)
print(f"Final Model Saved : {save_path}")

In [ ]:
checkpoint = torch.load(save_path, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

model.eval()

print("PPO model loaded successfully.")